# Exploration


In [ ]:
import pandas as pd
df=pd.read_csv("/kaggle/input/competitions/GiveMeSomeCredit/cs-training.csv")
print(df.head())
print(df.shape)
print(df.info())
print(df.describe())

In [ ]:
df = df.drop(columns=["Unnamed: 0"])


In [ ]:
df["age"].describe()
df[df["age"] < 18] #one entry with age=0, we just drop it
df = df[df["age"] > 0]

# Clip utilization to 500 %

In [ ]:

df["RevolvingUtilizationOfUnsecuredLines"] = df["RevolvingUtilizationOfUnsecuredLines"].clip(upper=5)

# ~~Clip NumberOfTime30-59DaysPastDueNotWorse to 6~~

# ~~Clip NumberOfTime60-89DaysPastDueNotWorse to 4~~

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap="coolwarm")
plt.show()


# Building the Train, Test and Val set

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

target = "SeriousDlqin2yrs"

X = df.drop(columns=[target])
y = df[target]

# Train & Val

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()Test

In [ ]:
df_test = pd.read_csv(
    "/kaggle/input/competitions/GiveMeSomeCredit/cs-test.csv"
)

df_test = df_test.drop(columns=["Unnamed: 0"])
df_test["RevolvingUtilizationOfUnsecuredLines"] = \
    df_test["RevolvingUtilizationOfUnsecuredLines"].clip(upper=5)
X_test = df_test.drop(columns=[target])

# Model

In [ ]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
}

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(df_test))
best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print(f"Fold {fold+1}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = lgb.LGBMClassifier(
        **params,
        n_estimators=5000
    )
    
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )
    best_iterations.append(model.best_iteration_)
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    

final_n_estimators = int(np.mean(best_iterations))
print("Average best iteration:", final_n_estimators)
# Overall CV AUC
auc = roc_auc_score(y, oof_preds)
print("CV AUC:", auc)

In [ ]:
final_model = lgb.LGBMClassifier(
    **params,
    n_estimators=final_n_estimators
)

final_model.fit(X, y)

In [ ]:
X_test = df_test.drop(columns=[target])
X_test = X_test[X.columns]

test_preds = final_model.predict_proba(X_test)[:, 1]

In [ ]:
submission = pd.DataFrame({
    "Id": df_test.index + 1,
    "Probability": test_preds
})

submission.to_csv("submission.csv", index=False)

